In [1]:
import numpy as np
import pandas as pd
import mne,os

In [2]:
trigger_info = {21: ["visual", "face", "male"], 
                22: ["visual", "face", "female"], 
                23: ["visual", "square", "square"], 
                24: ["auditory", "speech", "a"], 
                25: ["auditory", "speech", "o"], 
                26: ["auditory", "speech", "i"], 
                27: ["auditory", "music", "C"], 
                28: ["auditory", "music", "D"], 
                29: ["auditory", "music", "E"], 
                30: ["mix", "square-speech", "square-a"], 
                31: ["mix", "square-speech", "square-o"], 
                32: ["mix", "square-speech", "square-i"], 
                33: ["mix", "square-music", "square-C"], 
                34: ["mix", "square-music", "square-D"], 
                35: ["mix", "square-music", "square-E"], 
                36: ["mix", "face-speech", "male-a"], 
                37: ["mix", "face-speech", "male-o"], 
                38: ["mix", "face-speech", "male-i"], 
                39: ["mix", "face-speech", "female-a"], 
                40: ["mix", "face-speech", "female-o"], 
                41: ["mix", "face-speech", "female-i"], 
                42: ["mix", "face-music", "male-C"], 
                43: ["mix", "face-music", "male-D"], 
                44: ["mix", "face-music", "male-E"], 
                45: ["mix", "face-music", "female-C"], 
                46: ["mix", "face-music", "female-D"], 
                47: ["mix", "face-music", "female-E"], }


In [6]:
def read_file(subject, session):
    path = f"multi/data/derivatives/sub-{subject:02d}/ses-{session:02d}/task_250_asr_ica.set"
    if not os.path.exists(path):
        return None
    raw = mne.io.read_raw_eeglab(path, verbose=False, preload=True)
    #print(raw.info['chs'])
    #print(raw.info['ch_names'])
    events, event_id = mne.events_from_annotations(raw,verbose=False)
    id_t = {v: int(k) for k, v in event_id.items()}
    epochs = mne.Epochs(raw, events, event_id=event_id,baseline=None,
                        tmin=-0.4, tmax=10.0,preload=True,verbose=False,
                        event_repeated='drop')
    epochs_6 = epochs["6"]
    event_list = epochs.events[:,2]
    idx = np.where( event_list == event_id["6"])[0]
    trial_type = [id_t[event_list[i]] for i in idx+1]
    epochs_6.metadata = pd.DataFrame({
        "subject":subject,
        "session":session,
        "trial_type": trial_type,
        "modality": [trigger_info[t][0] for t in trial_type],
        "category": [trigger_info[t][1] for t in trial_type],
        "object": [trigger_info[t][2] for t in trial_type]
    })
    return epochs_6

In [7]:
epochs_list = []
for subject in range(1, 27):
    for session in [1, 2]:
        epochs = read_file(subject, session)
        if epochs is not None:
            #print(subject, session)
            epochs_list.append(epochs)
for ep in epochs_list:
    ep.events[:, 2] = 6
    ep.event_id = {"6": 6}
epochs_merged = mne.concatenate_epochs(epochs_list)
epochs_merged.save("all-epo.fif", overwrite=True)


Adding metadata with 6 columns
Adding metadata with 6 columns
Adding metadata with 6 columns
Adding metadata with 6 columns
Adding metadata with 6 columns
Adding metadata with 6 columns
Adding metadata with 6 columns
Adding metadata with 6 columns
Adding metadata with 6 columns
Adding metadata with 6 columns
Adding metadata with 6 columns
Adding metadata with 6 columns
Adding metadata with 6 columns
Adding metadata with 6 columns
Adding metadata with 6 columns
Adding metadata with 6 columns
Adding metadata with 6 columns
Adding metadata with 6 columns
Adding metadata with 6 columns
Adding metadata with 6 columns
Adding metadata with 6 columns
Adding metadata with 6 columns
Adding metadata with 6 columns
Adding metadata with 6 columns
Adding metadata with 6 columns
Adding metadata with 6 columns
Adding metadata with 6 columns
Adding metadata with 6 columns


/tmp/ipykernel_1429199/1339262904.py:11: RuntimeWarning: Concatenation of Annotations within Epochs is not supported yet. All annotations will be dropped.
  epochs_merged = mne.concatenate_epochs(epochs_list)


Adding metadata with 6 columns
5376 matching events found
No baseline correction applied


[PosixPath('/home/song-z/Project/all-epo.fif')]